In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [4]:
def rfeFeature(in_x,out_y,n):
        rfelist=[]
        
        log_model = LogisticRegression(solver='lbfgs')
        RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
       # NB = GaussianNB()
        DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
        svc_model = SVC(kernel = 'linear', random_state = 0)
        #knn = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        rfemodellist=[log_model,svc_model,RF,DT] 
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(estimator=i, n_features_to_select=n)
            #log_rfe = RFE(i, n)
            log_fit = log_rfe.fit(in_x, out_y)
            log_rfe_feature=log_fit.transform(in_x)
            rfelist.append(log_rfe_feature)
        return rfelist

In [5]:
def split_scalar(in_x,out_y):
        x_train, x_test, y_train, y_test = train_test_split(in_x, out_y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(x_train)
        X_test = sc.transform(x_test)
        
        return x_train, x_test, y_train, y_test

In [6]:
def cm_prediction(classifier,x_test):
     y_pred = classifier.predict(x_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,x_test,y_test,cm

In [7]:
def logistic(x_train,y_train,x_test):       

        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm      

In [8]:
def svm_linear(x_train,y_train,x_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [9]:
def svm_NL(x_train,y_train,x_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [10]:
def Navie(x_train,y_train,x_test):       

        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm         

In [11]:
def knn(x_train,y_train,x_test):
           

        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [12]:
def Decision(x_train,y_train,x_test):
        

        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm      

In [13]:
def random(x_train,y_train,x_test):
        

        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [14]:
def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','SVMNl',
                                                                                        'KNN','Navie','Decision','Random'])

    for number,idex in enumerate(rfedataframe.index):
        
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['SVMNl'][idex]=accsvmnl[number]
        rfedataframe['KNN'][idex]=accknn[number]
        rfedataframe['Navie'][idex]=accnav[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe

In [15]:
dataset1=pd.read_csv("prep.csv",index_col=None)
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)

in_x=df2.drop('classification_yes', axis=1)
out_y=df2['classification_yes']


rfelist=rfeFeature(in_x,out_y,3)  
rfelist

LogisticRegression()


C:\Users\Dayana\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Dayana\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_ite

SVC(kernel='linear', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)


[array([[1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 0.],
        ...,
        [1., 0., 1.],
        [0., 0., 1.],
        [0., 0., 0.]]),
 array([[0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.],
        ...,
        [0., 1., 0.],
        [0., 1., 1.],
        [0., 0., 1.]]),
 array([[ 3.07735602, 12.51815562,  4.70559701],
        [ 0.7       , 10.7       ,  4.70559701],
        [ 0.6       , 12.        ,  4.70559701],
        ...,
        [ 6.        ,  9.1       ,  3.4       ],
        [ 6.8       ,  8.5       ,  4.70559701],
        [ 1.        , 16.3       ,  4.9       ]]),
 array([[ 3.07735602, 38.86890244,  0.        ],
        [ 0.7       , 34.        ,  0.        ],
        [ 0.6       , 34.        ,  0.        ],
        ...,
        [ 6.        , 26.        ,  0.        ],
        [ 6.8       , 38.86890244,  0.        ],
        [ 1.        , 53.        ,  0.        ]])]

In [16]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

for i in rfelist:   
    x_train, x_test, y_train, y_test=split_scalar(i,out_y)   
    
        
    classifier,Accuracy,report,x_test,y_test,cm=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=svm_linear(x_train,y_train,x_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=svm_NL(x_train,y_train,x_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=knn(x_train,y_train,x_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=Navie(x_train,y_train,x_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=Decision(x_train,y_train,x_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=random(x_train,y_train,x_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

result

C:\Users\Dayana\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Dayana\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Dayana\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Dayana\AppData\Local\Temp\ip

,Logistic,SVMl,SVMNl,KNN,Navie,Decision,Random
Logistic,0.94,0.94,0.94,0.94,0.94,0.94,0.94
SVC,0.87,0.87,0.87,0.64,0.87,0.87,0.87
Random,0.94,0.95,0.94,0.93,0.86,0.91,0.94
DecisionTree,0.95,0.95,0.9,0.93,0.77,0.94,0.97
